# Cascade netem sweep (real channel)

Measures the effect of real classical-network conditions (delay, loss) isolated to exactly the Cascade reconciliation phase, using `run_real_channel_cascade_netem_trial` (added to `sweep_utils.py`). No SDC faults are injected -- this measures network sensitivity alone, separate from the fault-injection sweeps.

Prerequisite: all 6 steps in `17_cascade_netem_validation.ipynb` passed. If you haven't run that, don't run this.

Two grids, run separately (jitter is included as a parameter but not swept by default -- the earlier local single_axis_sweeps data found no significant jitter effect in the 5-50ms range, so delay and loss are the priority):
- **Delay**: 0, 1, 10, 25, 50, 100, 150, 200 ms
- **Loss**: 0, 1, 3, 5, 8, 10, 13, 15 %

n=5 replicates per condition, matching the replicate count already used elsewhere in this project's sweeps.

In [2]:
import sys, os, time
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as df
from qne.cascade.sweep_utils import run_real_channel_cascade_netem_trial
fablib = df.get_fablib()

SLICE_NAME = 'qfabric-bb84-2'
slice_obj = fablib.get_slice(name=SLICE_NAME)
print(f"Reconnected to slice '{SLICE_NAME}' (state: {slice_obj.get_state()})")

alice_node = slice_obj.get_node("alice")
bob_node = slice_obj.get_node("bob")
bob_ip = bob_node.get_interface(network_name="net_switch_bob").get_ip_addr()
ALICE_IFACE = alice_node.get_interface(network_name="net_alice_switch").get_device_name()
BOB_IFACE = bob_node.get_interface(network_name="net_switch_bob").get_device_name()
print(f"ALICE_IFACE={ALICE_IFACE}  BOB_IFACE={BOB_IFACE}  bob_ip={bob_ip}")

df.upload_project(slice_obj)  # push current code (including sweep_utils.py) before collecting anything

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid
Reconnected to slice 'qfabric-bb84-2' (state: StableOK)
ALICE_IFACE=enp7s0  BOB_IFACE=enp7s0  bob_ip=10.10.1.2

=== Uploading project (clean tarball) ===
  Uploading to alice...
  Uploading to bob...
  Uploading to switch...
  Upload complete (qne + validation + scenarios + p4 on every node)


In [4]:
import csv

# key0 metadata, from key_pairs_metadata.csv -- change if you want a
# different key pair, or extend to loop over multiple keys later.
with open(PROJECT_DIR / "results" / "key_pairs_metadata.csv") as f:
    key_meta = {row["index"]: row for row in csv.DictReader(f)}
KEY_INDEX = "0"
real_qber = float(key_meta[KEY_INDEX]["qber"])
k_pe = int(key_meta[KEY_INDEX]["k_pe"])
print(f"Using key{KEY_INDEX}: qber={real_qber}, k_pe={k_pe}")

# NOTE: run_real_channel_cascade_netem_trial defaults to
# results/bob_sifted_bits.json / results/alice_sifted_bits.json (the fixed
# canonical pair used by the existing real-channel sweeps in this repo).
# If those aren't actually key0, copy key0's files to those fixed names on
# both nodes first, or pass bob_key_json=/alice_key_json= explicitly below
# pointing at results/bob_sifted_bits_key0.json / alice_sifted_bits_key0.json.

N_REPLICATES = 5
DELAY_CONDITIONS_MS = [0, 1, 10, 25, 50, 100, 150, 200]
LOSS_CONDITIONS_PCT = [0, 1, 3, 5, 8, 10, 13, 15]

def cleanup_stale_processes():
    """Kill any leftover driver processes before each trial. Cheap insurance
    against exactly the kind of stale-process hang this project's netem
    validation ran into more than once."""
    bob_node.execute("sudo pkill -9 -f bob_cascade_driver 2>/dev/null; true", quiet=True)
    alice_node.execute("sudo pkill -9 -f alice_cascade_responder 2>/dev/null; true", quiet=True)

Using key0: qber=0.01282051282051282, k_pe=390


## Delay sweep

In [9]:
delay_output_path = str(PROJECT_DIR / "results" / "sdc_realchannel_cascade_netem_delay_sweep.csv")

if os.path.exists(delay_output_path):
    df_existing = pd.read_csv(delay_output_path)
    completed = set(zip(df_existing["cascade_delay_ms"], df_existing["run"]))
    print(f"Resuming: {len(completed)} trials already done")
else:
    completed = set()
    print("Starting fresh")

header_written = os.path.exists(delay_output_path)

for delay_ms in DELAY_CONDITIONS_MS:
    for run in range(N_REPLICATES):
        if (delay_ms, run) in completed:
            continue

        cleanup_stale_processes()
        seed = 42 + run
        result = run_real_channel_cascade_netem_trial(
            bob_node, alice_node, bob_ip, real_qber, run, seed,
            bob_iface=BOB_IFACE, alice_iface=ALICE_IFACE,
            cascade_delay_ms=delay_ms, k=k_pe,
            bob_key_json="results/bob_sifted_bits_key0.json",
            alice_key_json="results/alice_sifted_bits_key0.json",
        )
        result["fault_type"] = "cascade_netem_delay"

        row_df = pd.DataFrame([result])
        row_df.to_csv(delay_output_path, mode="a", header=not header_written, index=False)
        header_written = True

        print(f"  delay={delay_ms}ms, run={run}: non_convergent={result.get('non_convergent')}, "
              f"reconciliation_elapsed_seconds={result.get('reconciliation_elapsed_seconds')} [saved]")

print("\nDelay sweep complete.")
print(pd.read_csv(delay_output_path).groupby("cascade_delay_ms").size())

Starting fresh
  delay=0ms, run=0: non_convergent=False, reconciliation_elapsed_seconds=0.23823785781860352 [saved]
  delay=0ms, run=1: non_convergent=False, reconciliation_elapsed_seconds=0.24229121208190918 [saved]
  delay=0ms, run=2: non_convergent=False, reconciliation_elapsed_seconds=0.47684311866760254 [saved]
  delay=0ms, run=3: non_convergent=False, reconciliation_elapsed_seconds=0.3112821578979492 [saved]
  delay=0ms, run=4: non_convergent=False, reconciliation_elapsed_seconds=0.2357468605041504 [saved]
  delay=1ms, run=0: non_convergent=False, reconciliation_elapsed_seconds=0.2707860469818115 [saved]
  delay=1ms, run=1: non_convergent=False, reconciliation_elapsed_seconds=0.2707064151763916 [saved]
  delay=1ms, run=2: non_convergent=False, reconciliation_elapsed_seconds=0.5534310340881348 [saved]
  delay=1ms, run=3: non_convergent=False, reconciliation_elapsed_seconds=0.3597579002380371 [saved]
  delay=1ms, run=4: non_convergent=False, reconciliation_elapsed_seconds=0.2705757

## Loss sweep

In [14]:
loss_output_path = str(PROJECT_DIR / "results" / "sdc_realchannel_cascade_netem_loss_sweep.csv")

if os.path.exists(loss_output_path):
    df_existing = pd.read_csv(loss_output_path)
    completed = set(zip(df_existing["cascade_loss_pct"], df_existing["run"]))
    print(f"Resuming: {len(completed)} trials already done")
else:
    completed = set()
    print("Starting fresh")

header_written = os.path.exists(loss_output_path)

for loss_pct in LOSS_CONDITIONS_PCT:
    for run in range(N_REPLICATES):
        if (loss_pct, run) in completed:
            continue

        cleanup_stale_processes()
        seed = 42 + run
        result = run_real_channel_cascade_netem_trial(
            bob_node, alice_node, bob_ip, real_qber, run, seed,
            bob_iface=BOB_IFACE, alice_iface=ALICE_IFACE,
            cascade_delay_ms=delay_ms, k=k_pe,
            bob_key_json="results/bob_sifted_bits_key0.json",
            alice_key_json="results/alice_sifted_bits_key0.json",
        )
        result["fault_type"] = "cascade_netem_loss"

        row_df = pd.DataFrame([result])
        row_df.to_csv(loss_output_path, mode="a", header=not header_written, index=False)
        header_written = True

        print(f"  loss={loss_pct}%, run={run}: non_convergent={result.get('non_convergent')}, "
              f"reconciliation_elapsed_seconds={result.get('reconciliation_elapsed_seconds')} [saved]")

print("\nLoss sweep complete.")
print(pd.read_csv(loss_output_path).groupby("cascade_loss_pct").size())

Starting fresh
  loss=0%, run=0: non_convergent=False, reconciliation_elapsed_seconds=7.0417046546936035 [saved]
  loss=0%, run=1: non_convergent=False, reconciliation_elapsed_seconds=7.060755968093872 [saved]
  loss=0%, run=2: non_convergent=False, reconciliation_elapsed_seconds=14.886138439178467 [saved]
  loss=0%, run=3: non_convergent=False, reconciliation_elapsed_seconds=9.519534587860107 [saved]
  loss=0%, run=4: non_convergent=False, reconciliation_elapsed_seconds=7.041172742843628 [saved]
  loss=1%, run=0: non_convergent=False, reconciliation_elapsed_seconds=7.043066501617432 [saved]
  loss=1%, run=1: non_convergent=False, reconciliation_elapsed_seconds=7.041293382644653 [saved]
  loss=1%, run=2: non_convergent=False, reconciliation_elapsed_seconds=14.893433094024658 [saved]
  loss=1%, run=3: non_convergent=False, reconciliation_elapsed_seconds=9.518736124038696 [saved]
  loss=1%, run=4: non_convergent=False, reconciliation_elapsed_seconds=7.041821241378784 [saved]
  loss=3%, r

## Quick sanity check before any real analysis
Confirm reconciliation_elapsed_seconds actually shows the expected monotonic trend and that non_convergent/keys_match don't show anything alarming, before treating this data as trustworthy for the paper.

In [11]:
delay_df = pd.read_csv(delay_output_path)
print(delay_df.groupby("cascade_delay_ms")["reconciliation_elapsed_seconds"].agg(["mean", "std", "count"]))
print()
print("non_convergent counts:", delay_df["non_convergent"].value_counts().to_dict())
print("keys_match counts:", delay_df["keys_match"].value_counts(dropna=False).to_dict())

                      mean       std  count
cascade_delay_ms                           
0                 0.300880  0.103284      5
1                 0.345051  0.122706      5
10                0.742765  0.270413      5
25                1.404419  0.518216      5
50                2.504072  0.930679      5
100               4.705443  1.755119      5
150               6.906466  2.580529      5
200               9.106705  3.404680      5

non_convergent counts: {False: 40}
keys_match counts: {True: 40}


In [15]:
loss_df = pd.read_csv(loss_output_path)
print(loss_df.groupby("cascade_loss_pct")["reconciliation_elapsed_seconds"].agg(["mean","std","count"]))
print("non_convergent:", loss_df["non_convergent"].value_counts().to_dict())
print("keys_match:", loss_df["keys_match"].value_counts(dropna=False).to_dict())

                      mean       std  count
cascade_loss_pct                           
0.0               9.107276  3.084364     40
non_convergent: {False: 40}
keys_match: {True: 40}
